# 06 - VUS Dataset & Feature Construction

Builds a VUS-prioritization dataset from trio-assembled variants.

This notebook:
1. Loads assembled trio variants
2. Splits trusted labels (P/LP vs B/LB) from VUS (unlabeled target set)
3. Builds robust features with missingness indicators
4. Adds gene-level enrichment signals from trusted labels
5. Saves artifacts for model training and VUS ranking

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.trio_loader import TrioLoader
from src.features import build_features

DATA_DIR = ROOT / 'data'
REPORTS_DIR = ROOT / 'reports'
DATA_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

IN_PARQUET = DATA_DIR / 'trio_variants.parquet'
TRIOS = DATA_DIR / 'trios.tsv'
PHENO = DATA_DIR / 'phenotypes.tsv'

OUT_X_TRUSTED = DATA_DIR / 'vus_trusted_X.parquet'
OUT_Y_TRUSTED = DATA_DIR / 'vus_trusted_y.csv'
OUT_G_TRUSTED = DATA_DIR / 'vus_trusted_groups.csv'
OUT_X_VUS = DATA_DIR / 'vus_unlabeled_X.parquet'
OUT_META_VUS = DATA_DIR / 'vus_unlabeled_meta.csv'
OUT_GENE_STATS = DATA_DIR / 'vus_gene_stats.csv'

print('ROOT:', ROOT)

ROOT: C:\Users\lvrga\OneDrive\work\predtest


In [2]:
if IN_PARQUET.exists():
    df = pd.read_parquet(IN_PARQUET)
    print(f'Loaded assembled variants from {IN_PARQUET.name}: {df.shape}')
else:
    loader = TrioLoader(trios_path=TRIOS, phenotypes_path=PHENO, data_dir=DATA_DIR)
    df = loader.load_all()
    df.to_parquet(IN_PARQUET, index=False)
    print(f'Assembled variants generated and saved to {IN_PARQUET.name}: {df.shape}')

required = {'Classification', 'trio_id'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Missing required columns: {sorted(missing)}')

df.head(3)

Loaded assembled variants from trio_variants.parquet: (4800, 30)


,Sample,Gene,Classification,Condition_Description,Condition_ID,HGVS_Protein,HGVS_Coding,Transcripts,Pathogenicity_Score,Condition_Inheritance,...,ClinVar,Knowledgebase,Functional,trio_id,father_has_variant,mother_has_variant,proband_zygosity,inheritance_mode,proband_phenotype_hpo,affected
0,SYNTH01_proband,MUC4,Uncertain significance,,,p.Val3926Ala,c.11777T>C,NM_018406.7|NM_004532.6|NM_138297.5|ENST000004...,,,...,,,PaPI:0.0|CADD:1.85|REVEL:0.032|PolyPhen2:0.0,T01,False,False,Homozygosity,de_novo_hom,HP:0000365;HP:0000407,1
1,SYNTH01_proband,SEMA3D,Uncertain significance,,,p.Glu1657Val,c.4970A>T,NM_001282857.2|NM_019001.5|ENST00000392981.7|E...,,,...,,,PaPI:0.601|CADD:21.4|REVEL:0.021|PolyPhen2:0.0...,T01,False,False,Heterozygosity,de_novo,HP:0000365;HP:0000407,1
2,SYNTH01_proband,PCSK1,Likely pathogenic,Premature ovarian failure 9,C3810376,p.Arg720*,c.2158C>T,NM_001017975.6|NR_165455.1|ENST00000370425.8|E...,5.086478320357829,Autosomal recessive,...,,,PaPI:1.0|CADD:39.0,T01,False,False,Heterozygosity,de_novo,HP:0000365;HP:0000407,1


## Labeling Strategy

- **Trusted positives**: Pathogenic, Likely pathogenic
- **Trusted negatives**: Benign, Likely benign
- **Unlabeled target set**: Uncertain significance (VUS)

In [3]:
cls = df['Classification'].fillna('').str.lower().str.strip()

is_pos = cls.isin(['pathogenic', 'likely pathogenic'])
is_neg = cls.isin(['benign', 'likely benign'])
is_vus = cls.eq('uncertain significance')

trusted_df = df[is_pos | is_neg].copy()
vus_df = df[is_vus].copy()

y_trusted = is_pos[trusted_df.index].astype(int).rename('y')
groups_trusted = trusted_df['trio_id'].astype(str).rename('trio_id')

print('Trusted set shape :', trusted_df.shape)
print('VUS set shape     :', vus_df.shape)
print('Trusted positive rate:', round(float(y_trusted.mean()), 4))

if trusted_df.empty:
    raise ValueError('No trusted labels found (P/LP or B/LB).')
if vus_df.empty:
    print('Warning: no VUS rows found. Ranking notebook will be empty.')

Trusted set shape : (50, 30)
VUS set shape     : (4750, 30)
Trusted positive rate: 1.0


In [4]:
combined = pd.concat([trusted_df, vus_df], axis=0, ignore_index=True)
X_all, _ = build_features(combined)

n_trusted = len(trusted_df)
X_trusted = X_all.iloc[:n_trusted].copy()
X_vus = X_all.iloc[n_trusted:].copy()

raw_missing_cols = [
    'Functional', 'Population', 'Pathogenicity_Score', 'Sample_Coverage',
    'Sample_Coverage_Alternate', 'Criteria', 'Condition', 'Condition_Inheritance'
]
for col in raw_missing_cols:
    miss = combined.get(col, pd.Series('', index=combined.index)).fillna('').astype(str).str.strip().eq('')
    feat_name = f'miss_{col.lower()}'
    X_trusted[feat_name] = miss.iloc[:n_trusted].astype(int).values
    X_vus[feat_name] = miss.iloc[n_trusted:].astype(int).values

gene_col = combined.get('Gene', pd.Series('UNKNOWN', index=combined.index)).fillna('UNKNOWN').astype(str)
gene_trusted = gene_col.iloc[:n_trusted].copy()

stat_df = pd.DataFrame({'Gene': gene_trusted, 'y': y_trusted.values})
gene_stats = stat_df.groupby('Gene', dropna=False)['y'].agg(['count', 'sum']).rename(columns={'count': 'gene_n', 'sum': 'gene_pos'})
gene_stats['gene_pos_rate'] = (gene_stats['gene_pos'] + 1.0) / (gene_stats['gene_n'] + 2.0)

total_pos = max(1, int(y_trusted.sum()))
total_neg = max(1, int((1 - y_trusted).sum()))
gene_stats['gene_neg'] = gene_stats['gene_n'] - gene_stats['gene_pos']
gene_stats['gene_log_or'] = np.log(((gene_stats['gene_pos'] + 0.5) / (total_pos + 1.0)) / ((gene_stats['gene_neg'] + 0.5) / (total_neg + 1.0)))

gene_all = gene_col.reset_index(drop=True)
mapped = gene_all.map(gene_stats['gene_pos_rate']).fillna(gene_stats['gene_pos_rate'].mean())
mapped_log_or = gene_all.map(gene_stats['gene_log_or']).fillna(0.0)
mapped_n = gene_all.map(gene_stats['gene_n']).fillna(0).astype(float)

X_trusted['gene_pos_rate'] = mapped.iloc[:n_trusted].values
X_vus['gene_pos_rate'] = mapped.iloc[n_trusted:].values
X_trusted['gene_log_or'] = mapped_log_or.iloc[:n_trusted].values
X_vus['gene_log_or'] = mapped_log_or.iloc[n_trusted:].values
X_trusted['gene_seen_count'] = mapped_n.iloc[:n_trusted].values
X_vus['gene_seen_count'] = mapped_n.iloc[n_trusted:].values

hpo_text = combined.get('proband_phenotype_hpo', pd.Series('', index=combined.index)).fillna('').astype(str)
hpo_count = hpo_text.apply(lambda s: 0 if not s.strip() else len([x for x in s.split(';') if x.strip()]))
X_trusted['proband_hpo_count'] = hpo_count.iloc[:n_trusted].values
X_vus['proband_hpo_count'] = hpo_count.iloc[n_trusted:].values

print('Trusted X shape:', X_trusted.shape)
print('VUS X shape    :', X_vus.shape)
print('Feature overlap ok:', set(X_trusted.columns) == set(X_vus.columns))

Trusted X shape: (50, 38)
VUS X shape    : (4750, 38)
Feature overlap ok: True


In [5]:
X_trusted.to_parquet(OUT_X_TRUSTED, index=False)
y_trusted.to_frame().to_csv(OUT_Y_TRUSTED, index=False)
groups_trusted.to_frame().to_csv(OUT_G_TRUSTED, index=False)
X_vus.to_parquet(OUT_X_VUS, index=False)

meta_cols = ['trio_id', 'Sample', 'Gene', 'Chr', 'Position', 'Ref', 'Alt', 'Classification', 'inheritance_mode']
vus_meta = vus_df.reindex(columns=[c for c in meta_cols if c in vus_df.columns]).copy()
if 'row_id' not in vus_meta.columns:
    vus_meta['row_id'] = np.arange(len(vus_meta))
vus_meta.to_csv(OUT_META_VUS, index=False)

gene_stats.reset_index().to_csv(OUT_GENE_STATS, index=False)

print('Saved:')
print(' -', OUT_X_TRUSTED.name)
print(' -', OUT_Y_TRUSTED.name)
print(' -', OUT_G_TRUSTED.name)
print(' -', OUT_X_VUS.name)
print(' -', OUT_META_VUS.name)
print(' -', OUT_GENE_STATS.name)

Saved:
 - vus_trusted_X.parquet
 - vus_trusted_y.csv
 - vus_trusted_groups.csv
 - vus_unlabeled_X.parquet
 - vus_unlabeled_meta.csv
 - vus_gene_stats.csv


In [6]:
display(X_trusted.head(3))
display(y_trusted.value_counts().rename('count').to_frame())
display(gene_stats.sort_values('gene_pos_rate', ascending=False).head(10))

if not vus_df.empty:
    display(vus_meta.head(10))

,cadd_score,revel_score,papi_score,polyphen2_score,sift_score,gnomad_af,pathogenicity_score,sample_coverage,alt_depth,sample_coverage_reference,...,miss_pathogenicity_score,miss_sample_coverage,miss_sample_coverage_alternate,miss_criteria,miss_condition,miss_condition_inheritance,gene_pos_rate,gene_log_or,gene_seen_count,proband_hpo_count
0,39.0,NaN,1.0,NaN,NaN,0.000011,5.086478,56.026637,22.111003,33.984189,...,0,0,0,0,1,0,0.666667,-2.140066,1.0,2
1,30.0,0.954,1.0,1.0,0.0,0.000003,5.087792,97.477771,54.050580,42.315958,...,0,0,0,0,1,0,0.666667,-2.140066,1.0,2
2,39.0,NaN,1.0,NaN,NaN,0.000011,5.025153,55.648363,22.041619,34.254100,...,0,0,0,0,1,0,0.750000,-1.629241,2.0,2


,count
y,
1,50


,gene_n,gene_pos,gene_pos_rate,gene_neg,gene_log_or
Gene,,,,,
MUC4,7,7,0.888889,0,-0.530628
CCDC66,2,2,0.750000,0,-1.629241
HLA-DRB5,2,2,0.750000,0,-1.629241
GOLGA8T,2,2,0.750000,0,-1.629241
ZNF773,2,2,0.750000,0,-1.629241
ASTN2,1,1,0.666667,0,-2.140066
BBIP1,1,1,0.666667,0,-2.140066
ATF7IP,1,1,0.666667,0,-2.140066
CIC,1,1,0.666667,0,-2.140066


,trio_id,Sample,Gene,Classification,inheritance_mode,row_id
0,T01,SYNTH01_proband,MUC4,Uncertain significance,de_novo_hom,0
1,T01,SYNTH01_proband,SEMA3D,Uncertain significance,de_novo,1
3,T01,SYNTH01_proband,SUSD5,Uncertain significance,de_novo,2
4,T01,SYNTH01_proband,AHNAK2,Uncertain significance,de_novo,3
5,T01,SYNTH01_proband,F7,Uncertain significance,de_novo,4
6,T01,SYNTH01_proband,PRUNE2,Uncertain significance,de_novo_hom,5
7,T01,SYNTH01_proband,GOLIM4,Uncertain significance,de_novo,6
8,T01,SYNTH01_proband,ANKRD36,Uncertain significance,de_novo,7
9,T01,SYNTH01_proband,CRYBG3,Uncertain significance,de_novo,8
10,T01,SYNTH01_proband,NEK8,Uncertain significance,de_novo,9
